# Commitment depth: descriptive statistics and tests

**Commitment depth** `d` is how far into a trace's own reasoning its answer distribution stops
moving, as a fraction of the tokens that trace generated. `d = 0.3` means the answer was
settled after 30% of the reasoning.

This notebook produces every number quoted in the results section, and the tests behind them:

0. how deep commitment happens, overall and per model
1. do models differ from each other?
2. does depth depend on the **dataset**?
3. does depth track **question difficulty**?
4. do traces that end up **wrong** commit at a different depth than those that end up right?

The primary metric is **TVD** (`tvd.py`): total variation distance between the answer
distributions at a boundary and every later boundary, so both the spread of diagnoses and any
shift between them count as non-commitment. The base-entity-share metric (`convergence.py`) is
carried alongside as a robustness check.

**Every test in sections 2 to 4 is run separately per model**, then Benjamini-Hochberg
corrected across the 8 models. Traces are not pooled across models: models differ enormously
in depth, were swept on different case mixes, and each case contributes two traces, so pooled
observations are neither exchangeable nor independent and a pooled p-value would be hard to
interpret. The cost is that each test sees only 33 to 86 cases, so a null is weak evidence.

Nothing is recomputed here; it reads outputs already written by those two scripts.
Run from this directory.

In [ ]:
import json, os
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.stats import false_discovery_control, kruskal, spearmanr, wilcoxon

ART = Path(os.environ.get("DCTAX_DATA_ROOT", Path.cwd().parents[1] / "artifacts")) / "rollouts"
OUT = Path("out")

tvd = pd.read_csv(OUT / "tvd_per_trace.csv").rename(columns={"converge_token_frac_tvd": "d"})
prop = pd.read_csv(OUT / "convergence_per_trace.csv").rename(columns={"converge_token_frac": "d"})
print(f"{len(tvd)} traces (TVD), {len(prop)} traces (proportion)")

## 0. How deep does commitment happen?

Cumulative share of traces committed by each depth, then the median and interquartile
range of commitment depth per model. These are the descriptive figures quoted in the
results, and they precede every test below.

In [ ]:
pd.DataFrame({
    metric: {f"d < {c}": f"{(df['d'] < c).sum()} / {len(df)} = {100*(df['d'] < c).mean():.0f}%"
             for c in (0.25, 0.5, 0.75)}
    for metric, df in [("TVD", tvd), ("proportion", prop)]
})

In [ ]:
def describe(df, label):
    """Median and IQR of commitment depth per model, as reported."""
    g = df.groupby("model")["d"].describe()[["count", "25%", "50%", "75%"]]
    g.columns = ["n", "q1", "median", "q3"]
    g = g.sort_values("median")
    return pd.DataFrame({
        (label, "n"): g["n"].astype(int),
        (label, "Mdn"): g["median"].map("{:.2f}".format),
        (label, "IQR"): g.apply(lambda r: f"{r['q1']:.2f}-{r['q3']:.2f}", axis=1),
    })

desc = describe(tvd, "TVD").join(describe(prop, "proportion"))
desc.columns = pd.MultiIndex.from_tuples(desc.columns)
desc

## 1. Do models differ?

**Test:** Welch's t-test on every pair of models (unequal variance, since group sizes run 70 to
173 and spreads differ), Benjamini-Hochberg corrected across all 28 pairs. Computed by
`pairwise_tests()` and read back here.

Mann-Whitney is carried in the same file as a robustness column, since depth is bounded and
skewed; pairs where the two disagree are worth treating as borderline.

In [ ]:
pairs = pd.read_csv(OUT / "tvd_pairwise_tests.csv")
print(f"{pairs['significant'].sum()} / {len(pairs)} pairs significant at q < 0.05\n")
print("pairs that do NOT differ:")
print(pairs.loc[~pairs["significant"], ["model_a", "model_b", "p_welch_bh"]]
           .to_string(index=False))
print("\npairs where Mann-Whitney disagrees (borderline):")
print(pairs.loc[~pairs["agrees_with_mannwhitney"],
                ["model_a", "model_b", "p_welch_bh", "p_mannwhitney_bh"]].to_string(index=False))

## The question-level variables

| variable | how it is derived | grain |
| --- | --- | --- |
| `source` | dataset the case was drawn from, `pool_screen1000.json` | case |
| `wrong_frac` | `n_wrong / n_graded`, i.e. of the 8 screening rollouts for this case, the share **this model** got wrong | model x case |
| `case_wrong_frac` | `wrong_frac` averaged over every model that swept the case: how hard the case is for the roster as a whole | case |

Both difficulty proxies come from the **screening** stage (8 rollouts from the prompt, before
any sweep), so they are measured independently of the depth they are tested against.

In [ ]:
source = {r["case_id"]: r["source"]
          for r in json.loads((ART / "pool_screen1000.json").read_text())["rows"]}
screen = pd.DataFrame(json.loads((ART / "cases_screen1000.json").read_text())["rows"])
screen["wrong_frac"] = screen["n_wrong"] / screen["n_graded"]
case_diff = screen.groupby("case_id")["wrong_frac"].mean().rename("case_wrong_frac")

def build(df):
    return (df[["model", "case_id", "base_is_correct", "d"]]
            .assign(source=lambda x: x["case_id"].map(source))
            .merge(screen[["model", "case_id", "wrong_frac"]], on=["model", "case_id"])
            .merge(case_diff, on="case_id"))

df, df_prop = build(tvd), build(prop)
print(f"{len(df)} traces with a dataset and a screening record")
df.head(3)

## 2. Dataset

**Test:** Kruskal-Wallis across MedQA / MedMCQA / NEJM CPC, within each model. It is the
rank-based analogue of one-way ANOVA, used because depth is bounded in [0, 1], skewed, and
piles up at the endpoints.

In [ ]:
SRC = ["medqa", "medmcqa", "nejm_cpc"]

def dataset_test(d):
    rows = []
    for model, g in d.groupby("model"):
        groups = [g.loc[g["source"] == s, "d"] for s in SRC]
        rows.append({"model": model, "n": len(g),
                     **{s: grp.median() for s, grp in zip(SRC, groups)},
                     "p": kruskal(*[x for x in groups if len(x) >= 5]).pvalue})
    out = pd.DataFrame(rows)
    out["q_bh"] = false_discovery_control(out["p"], method="bh")
    return out.round(3)

res = dataset_test(df)
print(f"significant after BH: {(res['q_bh'] < 0.05).sum()} of {len(res)}")
print(f"models where NEJM CPC has the latest median: "
      f"{(res[SRC].idxmax(axis=1) == 'nejm_cpc').sum()} of {len(res)}")
res

## 3. Difficulty

**Test:** Spearman rank correlation, which measures monotonic association without assuming
linearity. If depth were a difficulty measure, harder cases should commit *later*, so a
positive rho.

`case_wrong_frac` is the variable to trust: it spans 0.12 to 0.76 across the 101 cases,
and being case-level it is not confounded with the model's own competence.

In [ ]:
def difficulty_test(d):
    rows = []
    for model, g in d.groupby("model"):
        own, case = spearmanr(g["wrong_frac"], g["d"]), spearmanr(g["case_wrong_frac"], g["d"])
        rows.append({"model": model, "n": len(g),
                     "rho_own": own.statistic, "p_own": own.pvalue,
                     "rho_case": case.statistic, "p_case": case.pvalue})
    out = pd.DataFrame(rows)
    for c in ("own", "case"):
        out[f"q_{c}"] = false_discovery_control(out[f"p_{c}"], method="bh")
    return out.round(3)

res = difficulty_test(df)
print(f"significant after BH: own {(res['q_own'] < 0.05).sum()}, "
      f"case-level {(res['q_case'] < 0.05).sum()}, of {len(res)}")
res

## 4. Correct vs wrong base traces

**Test:** Wilcoxon signed-rank, **paired within case**. Each case contributes one trace that
reached the right answer and one that did not, under the same model, so difficulty, phrasing
and gold answer are held fixed and only the sampled trajectory differs. Pairing is what the
design was built for, so an unpaired test would discard it.

A case enters only if both of its arms survived the analysis filters, which is why the pair
counts are below the trace counts above.

In [ ]:
def arm_test(d):
    wide = (d.pivot_table(index=["model", "case_id"], columns="base_is_correct", values="d")
             .dropna().reset_index())
    rows = []
    for model, g in wide.groupby("model"):
        rows.append({"model": model, "n_pairs": len(g),
                     "median_correct": g[True].median(), "median_wrong": g[False].median(),
                     "median_diff": (g[True] - g[False]).median(),
                     "p": wilcoxon(g[True], g[False]).pvalue})
    out = pd.DataFrame(rows)
    out["q_bh"] = false_discovery_control(out["p"], method="bh")
    return out.round(3), len(wide)

res, n_pairs = arm_test(df)
print(f"{n_pairs} case-model pairs; significant after BH: {(res['q_bh'] < 0.05).sum()} of {len(res)}")
res

## 5. How concentrated is the decline?

`tvd.py` also reports, for each fraction `q` of a trace's total decline in worst-case
TVD, the shortest window achieving it. The window is always **anchored at the commitment
boundary and measured backwards**, never a free search for the narrowest span anywhere in
the trace, so a transient dip cannot be mined for a narrow window.

`width_of_descent` expresses that window as a fraction of the tokens between boundary 0
and the commitment point. A perfectly even decline would give `w(q) = q`, so values well
below the diagonal mean the decline is concentrated.

### Exclusions

A trace has no descent to characterise when it never committed (the commitment point is
the fallback at the final boundary) or when it was already committed before its first
sentence. A third criterion in the code, worst-case TVD never falling between boundary 0
and commitment, catches nothing here.

In [ ]:
drop = pd.read_csv(OUT / "tvd_drop_width.csv")
kept = set(drop["trace_id"])

def status(r):
    if r["trace_id"] in kept:
        return "retained"
    if r["converge_boundary_tvd"] >= r["n_boundaries"] - 1:
        return "never committed"
    if r["converge_boundary_tvd"] == 0:
        return "committed at boundary 0"
    return "TVD did not fall"

excl = tvd.assign(status=tvd.apply(status, axis=1)).pivot_table(
    index="model", columns="status", aggfunc="size", fill_value=0)
excl["total"] = excl.sum(axis=1)
print(f"retained {len(kept)} / {len(tvd)} = {100*len(kept)/len(tvd):.1f}%")
excl

### Width of the window, as a fraction of the descent

Reported at `q = 0.8` and `q = 1.0`. Two aggregations are given. Taking the **model** as
the unit sidesteps the dependence between traces (each case contributes a correct and a
wrong arm, and recurs across models); taking the **trace** as the unit gives a tighter SE
that assumes an independence the data does not have.

In [ ]:
w = (drop[drop["drop_fraction"].isin([0.8, 1.0])]
     .pivot_table(index=["model", "trace_id"], columns="drop_fraction",
                  values="width_of_descent").reset_index())
w.columns = ["model", "trace_id", "w80", "w100"]

per = w.groupby("model").agg(n=("w80", "size"),
                             w80_mean=("w80", "mean"), w80_se=("w80", "sem"),
                             w100_mean=("w100", "mean"), w100_se=("w100", "sem"))
print("model as the unit (mean of the per-model means, SE over models):")
for c in ("w80_mean", "w100_mean"):
    print(f"  {c[:-5]:<5} {per[c].mean():.3f} +/- {per[c].sem():.3f}")
print("trace as the unit (grand mean):")
for c in ("w80", "w100"):
    print(f"  {c:<5} {w[c].mean():.3f} +/- {w[c].sem():.3f}   n={len(w)}")
per.round(3)

### Does it hold at every commitment depth?

The concentration is present in every band, but it is not constant: it strengthens with
later commitment. Early-committing traces have the least concentrated declines, partly
because a short descent contains few boundaries and so has a coarser resolution floor.

In [ ]:
w = w.merge(tvd[["trace_id", "d"]], on="trace_id")
w["band"] = pd.cut(w["d"], [0, 0.25, 0.5, 0.75, 1.01],
                   labels=["0-0.25", "0.25-0.5", "0.5-0.75", "0.75-1"], right=False)
w.groupby("band", observed=True).agg(
    n=("w80", "size"), w80_mean=("w80", "mean"), w80_se=("w80", "sem"),
    w100_mean=("w100", "mean"), w100_se=("w100", "sem")).round(3)

### The same windows in raw tokens

`width` is the window as a fraction of the whole trace's reasoning tokens, so multiplying
by `reasoning_tokens` gives the window in tokens. Raw counts are the more physical
statement ("the collapse takes N tokens") but they scale with trace length, so both are
worth reporting.

In [ ]:
tokens = pd.read_csv(OUT / "convergence_per_trace.csv")[["trace_id", "reasoning_tokens"]]
raw = (drop[drop["drop_fraction"].isin([0.8, 1.0])]
       .pivot_table(index=["model", "trace_id"], columns="drop_fraction", values="width")
       .reset_index())
raw.columns = ["model", "trace_id", "t80", "t100"]
raw = raw.merge(tokens, on="trace_id")
for c in ("t80", "t100"):
    raw[c] = raw[c] * raw["reasoning_tokens"]

def tokens_summary(g):
    out = {"n": len(g), "reasoning_Mdn": g["reasoning_tokens"].median()}
    for c, label in (("t80", "tok80"), ("t100", "tok100")):
        out[f"{label}_mean"] = g[c].mean()
        out[f"{label}_Mdn"] = g[c].median()
        out[f"{label}_IQR"] = f"{g[c].quantile(.25):.0f}-{g[c].quantile(.75):.0f}"
    return out

print("all retained traces:")
for k, v in tokens_summary(raw).items():
    print(f"  {k:<16} {v if isinstance(v, str) else round(v)}")
print()
pd.DataFrame([{"model": m, **tokens_summary(g)} for m, g in raw.groupby("model")]
             ).set_index("model").round(0)

### Raw tokens, split by model and commitment band

Some cells are thin: gemma contributes 5 traces to the `0.75-1` band and glm 18, so read
those rows as indicative. `reasoning_Mdn` is carried alongside because the raw token
counts scale with trace length, and it varies across bands within a model.

In [ ]:
raw = raw.merge(w[["trace_id", "band"]], on="trace_id")
by_band = pd.DataFrame([
    {"model": m, "band": b, **tokens_summary(g)}
    for (m, b), g in raw.groupby(["model", "band"], observed=True)
]).set_index(["model", "band"])
by_band.round(0)

## Robustness: the same three tests on the base-entity-share metric

In [ ]:
d_res = dataset_test(df_prop)
f_res = difficulty_test(df_prop)
a_res, n_pairs = arm_test(df_prop)
print(f"dataset:        {(d_res['q_bh'] < 0.05).sum()} of 8 significant after BH")
print(f"difficulty:     {(f_res['q_case'] < 0.05).sum()} of 8 (case-level proxy)")
print(f"correct/wrong:  {(a_res['q_bh'] < 0.05).sum()} of 8 ({n_pairs} pairs)")

## 7. What does the band $\delta$ actually buy?

A trace counts as committed at the first position whose worst forward TVD falls within
$\delta = 0.2$. That value was carried over from the earlier proportion-based rule rather
than derived, so it needs both an interpretation and a sensitivity check.

**What it means.** Total variation distance is the largest disagreement over any event,
$\mathrm{TVD}(P, Q) = \max_A |P(A) - Q(A)|$. Taking $A$ to be a single answer gives
$\mathrm{TVD} \geq |p_X - q_X|$, so

> $\delta = 0.2$ means no single diagnosis's share of the rollouts changes by more than 20
> percentage points between that position and any later position.

That is the defensible reading. It is a statement about the answer distribution being
**stable**, not about how many diagnoses remain in play, which is a property of a single
position rather than a relation between two. A trace sitting at a steady five-way split
satisfies the rule; a trace oscillating between two diagnoses violates it.

Run `delta_sweep.py` first; it writes `out/delta_sweep.csv`.

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
from dctax.config import load_resample  # noqa: E402

MODELS = list(load_resample("screen1000").models)
sweep = pd.read_csv(OUT / "delta_sweep.csv")
print(f"{sweep['trace_id'].nunique()} traces over "
      f"{sweep['delta'].nunique()} values of delta")

In [ ]:
def iqr(series, fmt="{:.3f}"):
    """Median with its interquartile range, as one cell of a table."""
    q1, med, q3 = series.quantile([0.25, 0.5, 0.75])
    return f"{fmt.format(med)} [{fmt.format(q1)}, {fmt.format(q3)}]"

rows = []
for delta, g in sweep.groupby("delta"):
    ok = g[g["commits"] == 1]
    rows.append({
        "delta": delta,
        "committing": f"{100 * g['commits'].mean():.1f}%",
        "depth median [IQR]": iqr(ok["depth"]),
        "before mid-trace": f"{100 * (ok['depth'] < 0.5).mean():.1f}%",
        "top answer share": iqr(ok["top_share"], "{:.2f}"),
        "second answer share": iqr(ok["second_share"], "{:.2f}"),
    })
table = pd.DataFrame(rows).set_index("delta")
print(table.to_string())
table.to_csv(OUT / "delta_sweep_table.csv")

The two rightmost columns describe the answer distribution at the moment a trace is called
committed, and they are the interpretable version of the question. At $\delta = 0.2$ the
median trace has 85% of its rollouts on one diagnosis and 10% on the next; at
$\delta = 0.05$ it has all of them on one. Tightening the band does not only move the
commitment point later, it changes what commitment means: from *one answer dominates* to
*one answer is the only answer*.

This also disposes of a plausible-sounding rationale worth not repeating: that
$\delta = 0.2$ is roughly where a model is "choosing between about two diagnoses". The
median count of distinct answers there is indeed 2, but the split is about 85/10 rather
than a coin flip, so calling it a two-way contest overstates the remaining uncertainty.

In [ ]:
at_default = sweep[(sweep["delta"] == 0.20) & (sweep["commits"] == 1)]
print(f"at delta = 0.2, over {len(at_default)} committing traces")
print(at_default[["top_share", "second_share", "top_two_share",
                  "n_distinct", "n_over_10pct", "effective_n"]]
      .describe(percentiles=[0.25, 0.5, 0.75]).round(3).to_string())
print(f"\n  one answer above half the mass:                 "
      f"{100 * (at_default['top_share'] > 0.5).mean():.1f}% of traces")
print(f"  a genuine two-way split (top share below 0.6): "
      f"{100 * (at_default['top_share'] < 0.6).mean():.1f}% of traces")

# A curve value sitting at delta means the trace only just qualified, so the threshold is
# doing real work rather than being comfortably cleared.
print("\nworst forward TVD actually attained at the commitment point")
print(at_default["tvd_at_commit"].describe(percentiles=[0.25, 0.5, 0.75, 0.9])
      .round(3).to_string())

**`effective_n`**, above, is $2^H$ for the Shannon entropy $H$ of the answer distribution:
the number of equally likely answers that would leave the model this uncertain. It is 1.0
when one answer holds all the mass and 2.0 for a true coin flip between two. It summarises
the whole tail in one number where the two shares do not, but the shares are the ones to
quote in text.

### Sensitivity

Commitment depth moves materially with $\delta$, so any number quoted from it has to carry
the band it was computed at: between $\delta = 0.05$ and $0.5$ the median depth runs from
0.86 to 0.28 and the share of traces committing at all from 70% to 99%.

What survives is the qualitative claim, that a substantial part of the trace follows
commitment at every band tested. The per-model ordering does not survive as cleanly, and
the rank correlation below says how far it drifts.

In [ ]:
by_model = (sweep[sweep["commits"] == 1]
            .pivot_table(index="model", columns="delta", values="depth", aggfunc="median")
            .reindex(MODELS))
print("median commitment depth")
print(by_model.round(3).to_string())

print("\nshare of traces committing at all (%)")
print((100 * sweep.pivot_table(index="model", columns="delta", values="commits",
                               aggfunc="mean")).reindex(MODELS).round(1).to_string())

print("\nrank correlation of the per-model ordering against the ordering at delta = 0.2")
for delta in by_model.columns:
    rho, _ = spearmanr(by_model[0.20], by_model[delta])
    print(f"  delta = {delta:.2f}: rho = {rho:+.3f}")

## Summary

| question | result (TVD) |
| --- | --- |
| Models differ? | **Yes**, 23 of 28 pairs significant |
| Dataset? | **Not per model.** 0 of 8 survive BH, though NEJM CPC has the latest median in 5 of 8 |
| Difficulty? | **Not as a property of the question.** 0 of 8 on the case-level proxy. The model's *own* wrong-fraction correlates weakly and positively in 2 of 8 (ds-qwen-1.5b, qwq-32b) |
| Right vs wrong trace? | **No.** 0 of 8 survive BH, median within-case difference near zero |

### Caveats

- **Each per-model test sees only 33 to 86 cases.** A null is weak evidence, and the dataset
  result is the clearest example: no model reaches significance, yet NEJM CPC has the latest
  median in 5 of 8 (a sixth, glm, ties it), which is a pattern the per-model tests are
  individually unable to detect.
  Report the direction count alongside the nulls rather than claiming no effect exists.
- **The two difficulty proxies disagree.** The case-level measure is null everywhere, but
  the model's own wrong-fraction on a case is weakly positive in ds-qwen-1.5b and qwq-32b.
  That is more readily read as competence than as difficulty: cases a given model struggles
  with take it longer to settle, which is not a property of the question.
- **The difficulty range is truncated by design.** A case only entered the sweep if the model
  produced both a correct and a wrong screening sample, so trivially easy and hopeless cases
  were filtered out before anything was measured.
- **Dataset is confounded with answer-vocabulary size.** NEJM cases carry more distinct
  diagnoses per case for nearly every model, and TVD is by construction sensitive to how many
  answers are in play.
- **gemma has n = 70 traces and 33 pairs**, roughly half the other models.